# negprobe on Colab (v3, repaired templates)

Self-contained: the package is written out below. **Runtime -> Change runtime type -> T4 GPU first.**

This version includes the `relations.jsonl` template fix (P103, P190). The four run cells reproduce the Pythia and Qwen runs reported in the paper (float32, Colab T4, 28 Sep 2026).

In [ ]:
!nvidia-smi -L
!pip -q install 'transformers>=4.44' accelerate bitsandbytes 2>&1 | tail -2

In [ ]:
import os, pathlib
os.makedirs('negprobe/negprobe', exist_ok=True)
pathlib.Path('negprobe/negprobe/__init__.py').write_text('')
files = {}

files['data.py'] = r'''
"""Load negated LAMA probes into one flat list of affirmative/negated pairs.

Two input layouts are supported:

1. Facebook's full negated LAMA release (``negated_data.tar.gz``):
   Google_RE / ConceptNet / Squad samples carry a ``negated`` field;
   T-REx samples are negated with ``template_negated`` from ``relations.jsonl``.
2. Kassner's LAMA_primed_negated repo (``data/<Subset>/<variant>/*.jsonl``):
   every sample carries ``masked_negations``. The negations are identical
   across the three misprime variants, so only one variant is read.
"""
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass, asdict
from pathlib import Path

MASK = "[MASK]"

# Everything after the mask must be punctuation for a causal LM to predict it
# as the next token (e.g. "... born in [MASK] ." or "(born [MASK]).").
_TAIL_OK = re.compile(r"^[\s\.\)\],;:!?\"']*$")


@dataclass
class Pair:
    subset: str
    relation: str
    uid: str
    sub: str
    obj: str
    pos: str  # affirmative sentence containing exactly one [MASK]
    neg: str  # negated sentence containing exactly one [MASK]

    @property
    def mask_final(self) -> bool:
        """True if [MASK] is the last content token in both sentences."""
        return all(_TAIL_OK.match(s.split(MASK, 1)[1]) for s in (self.pos, self.neg))

    def to_dict(self):
        d = asdict(self)
        d["mask_final"] = self.mask_final
        return d


def _join(sentences) -> str:
    if isinstance(sentences, str):
        return sentences.strip()
    return " ".join(s.strip() for s in sentences)


def _clean(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def _fill(template: str, sub: str) -> str:
    return template.replace("[X]", sub).replace("[Y]", MASK)


# The released relations.jsonl has an off-by-one error: P103 carries P127's
# negated template and P190 carries P103's, so for these two relations the
# "negated" sentence is a different statement rather than a negation.
# Detected by comparing each template with its negated counterpart; see
# check_templates(). The corrected templates are used unless the environment variable
# NEGPROBE_RELEASED_TEMPLATES=1 is set, which reproduces the released behaviour.
TEMPLATE_FIXES = {
    "P103": "The native language of [X] is not [Y] .",
    "P190": "[X] and [Y] are not twin cities .",
}
if os.environ.get("NEGPROBE_RELEASED_TEMPLATES") == "1":
    TEMPLATE_FIXES = {}

# Google-RE has no negated field and no relations.jsonl entry; these are the
# templates hard-coded in LAMA's scripts/run_experiments.py.
GOOGLE_RE_TEMPLATES = {
    "place_of_birth": ("[X] was born in [Y] .", "[X] was not born in [Y] ."),
    "date_of_birth": ("[X] (born [Y]).", "[X] (not born [Y])."),
    "place_of_death": ("[X] died in [Y] .", "[X] did not die in [Y] ."),
}


def _canonical_subset(name: str) -> str:
    n = name.lower().replace("-", "").replace("_", "")
    if n.startswith("trex"):
        return "TREx"
    if n.startswith("googlere"):
        return "GoogleRE"
    if n.startswith("conceptnet"):
        return "ConceptNet"
    if n.startswith("squad"):
        return "SQuAD"
    return name


def _read_jsonl(path: Path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def _make_pair(subset, relation, sample, pos, neg, idx) -> Pair | None:
    if not pos or not neg:
        return None
    pos, neg = _clean(pos), _clean(neg)
    # keep only the first mask, as LAMA does ("score only first mask")
    if pos.count(MASK) < 1 or neg.count(MASK) < 1:
        return None
    uid = str(sample.get("uuid", sample.get("id", idx)))
    return Pair(
        subset=subset,
        relation=relation,
        uid=uid,
        sub=str(sample.get("sub_label", "")),
        obj=str(sample["obj_label"]).strip(),
        pos=pos,
        neg=neg,
    )


def load_kassner_repo(root: Path, variant: str = "random") -> list[Pair]:
    """Read ``LAMA_primed_negated/data``."""
    pairs = []
    for subset_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        vdir = subset_dir / variant
        if not vdir.exists():
            continue
        subset = _canonical_subset(subset_dir.name)
        for f in sorted(vdir.glob("*.jsonl")):
            for i, s in enumerate(_read_jsonl(f)):
                relation = s.get("pred", f.stem)
                p = _make_pair(subset, relation, s,
                               _join(s["masked_sentences"]),
                               _join(s.get("masked_negations", [])), i)
                if p:
                    pairs.append(p)
    return pairs


def check_templates(templates: dict, verbose: bool = True) -> list[str]:
    """Flag relations whose negated template is not the affirmative one plus a negation.

    Returns the list of suspicious relation ids. Uses token overlap, so it catches
    a swapped template but not a subtly reworded one.
    """
    def strip_neg(t):
        t = re.sub(r"\b(not|never|n't)\b", " ", t)
        t = re.sub(r"\b(does|did|do)\b", " ", t)
        return set(re.sub(r"\s+", " ", t).lower().strip(" .").split())

    bad = []
    for rel, r in templates.items():
        a, b = strip_neg(r["template"]), strip_neg(r["template_negated"])
        if len(a | b) and len(a & b) / len(a | b) < 0.5:
            bad.append(rel)
    if bad and verbose:
        print(f"warning: negated template looks wrong for {sorted(bad)} "
              f"(known issue in the released relations.jsonl)")
    return bad


def load_facebook_release(root: Path, fix_templates: bool = True) -> list[Pair]:
    """Read the extracted ``negated_data.tar.gz`` (folder with one dir per subset).

    The layout is detected loosely, so small differences in folder names do
    not matter. T-REx needs a ``relations.jsonl`` somewhere under ``root``.
    """
    templates = {}
    for rel_file in root.rglob("relations.jsonl"):
        for r in _read_jsonl(rel_file):
            if r.get("template") and r.get("template_negated"):
                templates[r["relation"]] = dict(r)
    check_templates(templates)
    if fix_templates:
        for rel, fixed in TEMPLATE_FIXES.items():
            if rel in templates:
                templates[rel]["template_negated"] = fixed

    pairs = []
    for f in sorted(root.rglob("*.jsonl")):
        if f.name == "relations.jsonl":
            continue
        top = f.relative_to(root).parts[0]
        subset = _canonical_subset(top)
        relation = f.stem.replace("_test", "")
        for i, s in enumerate(_read_jsonl(f)):
            pos = _join(s.get("masked_sentences", []))
            rel_id = s.get("predicate_id", relation)
            if "negated" in s and s["negated"]:
                # ConceptNet and SQuAD carry the negated sentence directly
                neg = _join(s["negated"])
            elif "masked_negations" in s:
                neg = _join(s["masked_negations"])
            elif rel_id in templates and s.get("sub_label"):
                # T-REx: build both sides from the relation templates so that
                # they differ only in the negation (as LAMA does)
                pos = _fill(templates[rel_id]["template"], s["sub_label"])
                neg = _fill(templates[rel_id]["template_negated"], s["sub_label"])
            elif relation in GOOGLE_RE_TEMPLATES and s.get("sub_label"):
                pos_t, neg_t = GOOGLE_RE_TEMPLATES[relation]
                pos, neg = _fill(pos_t, s["sub_label"]), _fill(neg_t, s["sub_label"])
            else:
                continue
            relation_name = s.get("pred", relation)
            if str(relation_name).startswith("/"):  # Google-RE Freebase ids
                relation_name = relation
            p = _make_pair(subset, relation_name, s, pos, neg, i)
            if p:
                pairs.append(p)
    return pairs


def load_pairs(path: str | Path, subsets: list[str] | None = None,
               trex_templates: str | Path | None = None,
               fix_templates: bool = True) -> list[Pair]:
    """Auto-detect the layout and return all pairs.

    ``trex_templates``: optional LAMA ``relations.jsonl`` (affirmative + negated
    templates). When given, T-REx pairs are rebuilt from the templates so that
    affirmative and negated sentences differ only in the negation.
    """
    root = Path(path)
    is_kassner = any((d / "random").is_dir() for d in root.iterdir() if d.is_dir())
    pairs = (load_kassner_repo(root) if is_kassner
             else load_facebook_release(root, fix_templates=fix_templates))

    if trex_templates:
        tmpl = {r["relation"]: dict(r) for r in _read_jsonl(Path(trex_templates))}
        if fix_templates:
            for rel, fixed in TEMPLATE_FIXES.items():
                if rel in tmpl:
                    tmpl[rel]["template_negated"] = fixed
        for p in pairs:
            r = tmpl.get(p.relation)
            if p.subset == "TREx" and r and r.get("template_negated") and p.sub:
                p.pos = _fill(r["template"], p.sub)
                p.neg = _fill(r["template_negated"], p.sub)

    if subsets:
        wanted = {_canonical_subset(s) for s in subsets}
        pairs = [p for p in pairs if p.subset in wanted]

    # de-duplicate (same relation + uid)
    seen, out = set(), []
    for p in pairs:
        key = (p.subset, p.relation, p.uid, p.pos, p.neg)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def load_vocab(path: str | Path | None) -> list[str] | None:
    """LAMA common vocabulary (``common_vocab_cased.txt``), one token per line."""
    if not path:
        return None
    with open(path, encoding="utf-8") as f:
        return [w.strip() for w in f if w.strip()]
'''

files['metrics.py'] = r'''
"""Per-pair metrics and aggregation."""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch


def _ranks(x: torch.Tensor) -> torch.Tensor:
    """Row-wise ranks (0 = smallest). Ties are practically absent in float logits."""
    order = x.argsort(dim=-1)
    r = torch.empty_like(order, dtype=torch.float32)
    src = torch.arange(x.shape[-1], device=x.device, dtype=torch.float32).expand_as(order)
    r.scatter_(-1, order, src)
    return r


def spearman_rows(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Spearman rho between matching rows of a and b (same as scipy.stats.spearmanr
    when there are no ties). Computed on the GPU, which matters for 150k vocabularies."""
    ra, rb = _ranks(a), _ranks(b)
    ra = ra - ra.mean(-1, keepdim=True)
    rb = rb - rb.mean(-1, keepdim=True)
    return (ra * rb).sum(-1) / (ra.norm(dim=-1) * rb.norm(dim=-1))


def rank_of(dist: torch.Tensor, pos: int | None) -> float:
    """1-based rank of vocabulary position ``pos`` in ``dist`` (higher logp = better)."""
    if pos is None:
        return float("nan")
    return float((dist > dist[pos]).sum().item() + 1)


def exact_match_p1(df: pd.DataFrame) -> pd.DataFrame:
    """P@1 as in LAMA: the top-ranked candidate *is* the gold object (whole word).

    For masked models this equals ``gold rank == 1``. For causal models the rank is
    taken on the gold object's first token, which would also count a hit when only a
    prefix matches ("Green" for "Greenland") and, in bfloat16, when scores tie; the
    exact-match definition avoids both and is defined for every pair.
    """
    df = df.copy()
    gold = df["obj"].astype(str).str.strip()
    df["p1_pos"] = (df["top1_pos"].astype(str).str.strip() == gold).astype(float)
    df["p1_neg"] = (df["top1_neg"].astype(str).str.strip() == gold).astype(float)
    return df


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-pair results.

    Macro averages as in Petroni et al. (2019): mean over the pairs of each relation,
    then mean over relations within a subset. Micro averages over all pairs are
    reported too.
    """
    cols = {
        "rho": "spearman_rho",
        "overlap": "top1_overlap",
        "delta": "delta_gold_logp",
        "rank_drop": "gold_rank_dropped",
        "p1_pos": "p@1_affirmative",
        "p1_neg": "p@1_negated",
    }
    rows = []
    for subset, g in df.groupby("subset"):
        per_rel = g.groupby("relation")[list(cols)].mean()
        row = {"subset": subset, "n_pairs": len(g), "n_relations": len(per_rel)}
        for c, name in cols.items():
            row[f"{name}"] = per_rel[c].mean()
            row[f"{name}_micro"] = g[c].mean()
        # 95% bootstrap CIs over pairs (they belong to the *_micro means)
        row["spearman_rho_micro_ci95"] = _bootstrap(g, "rho")
        row["delta_micro_ci95"] = _bootstrap(g, "delta")
        rows.append(row)
    return pd.DataFrame(rows)


def _bootstrap(g: pd.DataFrame, col: str, n: int = 1000, seed: int = 0) -> str:
    vals = g[col].dropna().to_numpy()
    if len(vals) < 2:
        return ""
    rng = np.random.default_rng(seed)
    means = [rng.choice(vals, len(vals)).mean() for _ in range(n)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    return f"[{lo:.3f}, {hi:.3f}]"
'''

files['scorers.py'] = r'''
"""Model wrappers that return, for a batch of cloze sentences:

* the log-probability distribution over the (restricted) vocabulary at the
  [MASK] position  -> used for Spearman rho and rank-1 overlap (Experiment A)
* the log-probability of the gold object at that position
  -> used for the negation shift Delta (Experiment B)

Masked LMs (BERT, RoBERTa, ...) predict the [MASK] token directly.
Causal LMs (Pythia, Llama, Qwen, ...) predict the token after the text that
precedes [MASK]; only sentences where [MASK] is the last content token are
valid for them (see ``Pair.mask_final``).
"""
from __future__ import annotations

import math

import torch
from transformers import (AutoModelForCausalLM, AutoModelForMaskedLM,
                          AutoTokenizer)

from .data import MASK


def _check_finite(logits, model_name: str):
    """A checkpoint stored in fp16 overflows to NaN on CPU/MPS; fail loudly instead
    of writing a file full of meaningless ranks."""
    if not torch.isfinite(logits).all():
        raise RuntimeError(
            f"{model_name}: the model produced NaN/inf logits. This usually means a "
            "half-precision checkpoint is running on CPU or MPS. Re-run with --dtype fp32.")


def _dtype(name: str):
    return {"auto": "auto", "fp32": torch.float32, "fp16": torch.float16,
            "bf16": torch.bfloat16}[name]


class _Base:
    is_causal = False

    def __init__(self, model_name, device="auto", dtype="auto", vocab=None,
                 load_in_4bit=False, trust_remote_code=False):
        self.name = model_name
        self.tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote_code)
        kwargs = dict(trust_remote_code=trust_remote_code)
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
            kwargs["device_map"] = "auto"
        else:
            # "auto" would keep a checkpoint stored in fp16, which overflows to NaN
            # on CPU and MPS. Only use half precision when there is a CUDA GPU.
            if dtype == "auto" and not torch.cuda.is_available():
                dtype = "fp32"
            kwargs["dtype"] = _dtype(dtype)
            if device == "auto":
                kwargs["device_map"] = "auto" if torch.cuda.is_available() else None
        try:
            self.model = self._model_cls().from_pretrained(model_name, **kwargs).eval()
        except TypeError:  # transformers < 4.56 spells it torch_dtype
            if "dtype" in kwargs:
                kwargs["torch_dtype"] = kwargs.pop("dtype")
            self.model = self._model_cls().from_pretrained(model_name, **kwargs).eval()
        if not load_in_4bit and device not in ("auto",):
            self.model.to(device)
        self.device = next(self.model.parameters()).device
        self.lowercase = bool(getattr(self.tok, "do_lower_case", False))
        self.index_list = self._build_index_list(vocab)

    # ---- vocabulary -------------------------------------------------------
    def _word_ids(self, word: str) -> list[int]:
        raise NotImplementedError

    def _build_index_list(self, vocab):
        """Token ids the distribution is restricted to.

        With a LAMA common vocab: ids of the words that are a single token for
        this model (words that are not are dropped, as in LAMA).
        Without: the whole vocabulary minus special tokens.
        """
        if vocab:
            ids, seen = [], set()
            for w in vocab:
                t = self._word_ids(w.lower() if self.lowercase else w)
                if len(t) == 1 and t[0] not in seen:
                    seen.add(t[0])
                    ids.append(t[0])
            idx = torch.tensor(ids)
        else:
            special = set(self.tok.all_special_ids)
            n = self.model.get_output_embeddings().weight.shape[0]
            idx = torch.tensor([i for i in range(n) if i not in special])
        self._pos_in_index = {int(t): i for i, t in enumerate(idx)}
        return idx.to(self.device)

    def decode(self, token_id: int) -> str:
        return self.tok.decode([token_id]).strip()

    def gold_ids(self, obj: str) -> list[int]:
        return self._word_ids(obj.lower() if self.lowercase else obj)

    def restricted_position(self, token_id: int):
        """Position of ``token_id`` in the restricted distribution, or None."""
        return self._pos_in_index.get(int(token_id))

    def _prep(self, s: str) -> str:
        return s.lower().replace(MASK.lower(), MASK) if self.lowercase else s


class MaskedLMScorer(_Base):
    def _model_cls(self):
        return AutoModelForMaskedLM

    def _word_ids(self, word):
        # RoBERTa-style BPE marks a preceding space inside the token
        if self.tok.mask_token == "<mask>":
            return self.tok.encode(" " + word, add_special_tokens=False)
        return self.tok.encode(word, add_special_tokens=False)

    @torch.no_grad()
    def score(self, sentences: list[str], objs: list[str]):
        """Returns (restricted log-prob matrix [B, V'], gold log-probs [B])."""
        texts = []
        for s in sentences:
            s = self._prep(s)
            head, tail = s.split(MASK, 1)
            tail = tail.replace(MASK, self.tok.mask_token)  # later masks stay masked
            if self.tok.mask_token == "<mask>":
                head = head.rstrip()  # "<mask>" absorbs the preceding space
                texts.append(head + self.tok.mask_token + tail)
            else:
                texts.append(head + self.tok.mask_token + tail)
        enc = self.tok(texts, return_tensors="pt", padding=True, truncation=True,
                       max_length=512).to(self.device)
        logits = self.model(**enc).logits
        mask_pos = (enc["input_ids"] == self.tok.mask_token_id).int().argmax(dim=1)
        rows = logits[torch.arange(len(texts), device=self.device), mask_pos]
        _check_finite(rows, self.name)
        logp = torch.log_softmax(rows.float(), dim=-1)

        gold = torch.full((len(texts),), float("nan"))
        for i, o in enumerate(objs):
            ids = self.gold_ids(o)
            if len(ids) == 1:  # multi-token objects are skipped, as in LAMA
                gold[i] = logp[i, ids[0]].item()
        return logp[:, self.index_list], gold


class CausalLMScorer(_Base):
    is_causal = True

    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token

    def _model_cls(self):
        return AutoModelForCausalLM

    def _word_ids(self, word):
        return self.tok.encode(" " + word, add_special_tokens=False)

    def _prefix(self, s):
        return self._prep(s).split(MASK, 1)[0].rstrip()

    @torch.no_grad()
    def score(self, sentences: list[str], objs: list[str]):
        """Next-token distribution after the prefix, and the summed
        log-probability of the full (possibly multi-token) gold object."""
        prefixes = [self._prefix(s) for s in sentences]
        bos = [self.tok.bos_token_id] if self.tok.bos_token_id is not None else []
        seqs, starts, gold_lens = [], [], []
        for p, o in zip(prefixes, objs):
            p_ids = bos + self.tok.encode(p, add_special_tokens=False)
            o_ids = self.gold_ids(o)
            seqs.append(p_ids + o_ids)
            starts.append(len(p_ids))
            gold_lens.append(len(o_ids))

        # left-pad by hand so position bookkeeping stays simple
        L = max(map(len, seqs))
        pad = self.tok.pad_token_id
        input_ids = torch.full((len(seqs), L), pad, dtype=torch.long)
        attn = torch.zeros((len(seqs), L), dtype=torch.long)
        offsets = []
        for i, s in enumerate(seqs):
            off = L - len(s)
            input_ids[i, off:] = torch.tensor(s)
            attn[i, off:] = 1
            offsets.append(off)
        input_ids, attn = input_ids.to(self.device), attn.to(self.device)
        # with left padding, positions must be counted from the first real token
        position_ids = (attn.cumsum(-1) - 1).clamp_min(0)
        try:
            logits = self.model(input_ids=input_ids, attention_mask=attn,
                                position_ids=position_ids).logits.float()
        except TypeError:  # a few architectures do not take position_ids
            logits = self.model(input_ids=input_ids, attention_mask=attn).logits.float()
        _check_finite(logits, self.name)
        logp_all = torch.log_softmax(logits, dim=-1)

        next_rows, gold = [], torch.zeros(len(seqs))
        for i in range(len(seqs)):
            first = offsets[i] + starts[i]           # position of first gold token
            next_rows.append(logp_all[i, first - 1])  # prediction for that position
            lp = 0.0
            for k in range(gold_lens[i]):
                tok_id = input_ids[i, first + k]
                lp += logp_all[i, first + k - 1, tok_id].item()
            gold[i] = lp if gold_lens[i] else math.nan
        next_rows = torch.stack(next_rows)
        return next_rows[:, self.index_list], gold


def load_scorer(model_name: str, kind: str = "auto", **kw):
    """``kind``: 'mlm', 'causal' or 'auto' (decided from the model config)."""
    if kind == "auto":
        from transformers import AutoConfig
        cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=kw.get("trust_remote_code", False))
        archs = " ".join(cfg.architectures or []).lower()
        kind = "mlm" if ("masked" in archs or cfg.model_type in
                         {"bert", "roberta", "distilbert", "albert", "electra",
                          "xlm-roberta", "deberta", "deberta-v2", "modernbert"}) else "causal"
    return (MaskedLMScorer if kind == "mlm" else CausalLMScorer)(model_name, **kw)
'''

files['run.py'] = r'''
"""Run Experiments A and B for one model.

Example:
    python -m negprobe.run --model bert-base-cased --data data/negated_data \
        --vocab data/common_vocab_cased.txt --out results
"""
from __future__ import annotations

import argparse
import json
import math
import re
import time
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm

from .data import load_pairs, load_vocab
from .metrics import exact_match_p1, rank_of, spearman_rows, summarize
from .scorers import load_scorer


def slug(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", name)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True, help="Hugging Face model id or local path")
    ap.add_argument("--kind", default="auto", choices=["auto", "mlm", "causal"])
    ap.add_argument("--data", required=True, help="negated LAMA folder (Facebook release or LAMA_primed_negated/data)")
    ap.add_argument("--subsets", nargs="*", default=None, help="e.g. TREx GoogleRE ConceptNet SQuAD")
    ap.add_argument("--trex-templates", default=None, help="optional relations.jsonl with template/template_negated")
    ap.add_argument("--vocab", default=None, help="LAMA common_vocab_cased.txt (recommended for comparability)")
    ap.add_argument("--only-mask-final", action="store_true",
                    help="keep only pairs usable by causal LMs (always on for causal models)")
    ap.add_argument("--limit", type=int, default=None, help="use only the first N pairs (for testing)")
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--dtype", default="auto", choices=["auto", "fp32", "fp16", "bf16"])
    ap.add_argument("--load-in-4bit", action="store_true")
    ap.add_argument("--trust-remote-code", action="store_true")
    ap.add_argument("--out", default="results")
    args = ap.parse_args(argv)

    pairs = load_pairs(args.data, args.subsets, args.trex_templates)
    vocab = load_vocab(args.vocab)
    scorer = load_scorer(args.model, args.kind, device=args.device, dtype=args.dtype,
                         vocab=vocab, load_in_4bit=args.load_in_4bit,
                         trust_remote_code=args.trust_remote_code)
    if scorer.is_causal or args.only_mask_final:
        before = len(pairs)
        pairs = [p for p in pairs if p.mask_final]
        print(f"kept {len(pairs)}/{before} pairs where [MASK] is sentence-final")
    if args.limit:
        pairs = pairs[: args.limit]
    print(f"{args.model}: {len(pairs)} pairs, restricted vocab size {len(scorer.index_list)}")

    out_dir = Path(args.out) / slug(args.model)
    out_dir.mkdir(parents=True, exist_ok=True)
    records = []
    t0 = time.time()
    for i in tqdm(range(0, len(pairs), args.batch_size)):
        batch = pairs[i: i + args.batch_size]
        objs = [p.obj for p in batch]
        d_pos, g_pos = scorer.score([p.pos for p in batch], objs)
        d_neg, g_neg = scorer.score([p.neg for p in batch], objs)
        rho = spearman_rows(d_pos, d_neg).cpu()
        top_pos = d_pos.argmax(-1).cpu()
        top_neg = d_neg.argmax(-1).cpu()
        for j, p in enumerate(batch):
            gids = scorer.gold_ids(p.obj)
            # rank is defined on the first gold token (the only one for MLMs)
            gpos = scorer.restricted_position(gids[0]) if gids else None
            if not scorer.is_causal and len(gids) != 1:
                gpos = None
            r_pos, r_neg = rank_of(d_pos[j], gpos), rank_of(d_neg[j], gpos)
            gp, gn = g_pos[j].item(), g_neg[j].item()
            records.append({
                **p.to_dict(),
                "model": args.model,
                "rho": rho[j].item(),
                "overlap": int(top_pos[j] == top_neg[j]),
                "top1_pos": scorer.decode(scorer.index_list[top_pos[j]].item()),
                "top1_neg": scorer.decode(scorer.index_list[top_neg[j]].item()),
                "gold_logp_pos": gp,
                "gold_logp_neg": gn,
                "delta": gn - gp if not (math.isnan(gp) or math.isnan(gn)) else float("nan"),
                "gold_rank_pos": r_pos,
                "gold_rank_neg": r_neg,
                "rank_drop": float(r_neg > r_pos) if not math.isnan(r_pos) else float("nan"),
                "p1_pos": float(r_pos == 1) if not math.isnan(r_pos) else float("nan"),
                "p1_neg": float(r_neg == 1) if not math.isnan(r_neg) else float("nan"),
            })

    df = exact_match_p1(pd.DataFrame(records))  # P@1 = top-1 candidate is the gold object
    df.to_json(out_dir / "pairs.jsonl", orient="records", lines=True, force_ascii=False)
    summary = summarize(df)
    summary.insert(0, "model", args.model)
    summary.to_csv(out_dir / "summary.csv", index=False)
    meta = {"model": args.model, "kind": "causal" if scorer.is_causal else "mlm",
            "n_pairs": len(df), "vocab_size": len(scorer.index_list),
            "common_vocab": bool(vocab), "seconds": round(time.time() - t0, 1),
            "args": vars(args), "torch": torch.__version__}
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print(summary[["subset", "n_pairs", "n_relations", "spearman_rho", "top1_overlap",
                       "delta_gold_logp", "gold_rank_dropped", "p@1_affirmative"]].round(3))
    print(f"saved to {out_dir}")


if __name__ == "__main__":
    main()
'''

files['truth.py'] = r'''
"""Experiment C: do models judge negated statements correctly?

For every fact a 2x2 design is built:

    polarity (affirmative / negated)  x  object (gold / distractor)

    Albert Einstein was born in Ulm.          -> True
    Albert Einstein was not born in Ulm.      -> False
    Albert Einstein was born in Lyon.         -> False
    Albert Einstein was not born in Lyon.     -> True

The model answers True/False; the answer is read from the probabilities of the
True and False tokens, so nothing has to be parsed and the result is deterministic.
Instruction-tuned models are prompted through their chat template, base models
with a few fixed examples.

    python -m negprobe.truth --model Qwen/Qwen2.5-7B-Instruct --data data/negated_data \
        --subsets TREx --per-relation 100

Causal LMs only (masked LMs do not produce a continuation to score).
"""
from __future__ import annotations

import argparse
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from .data import MASK, load_pairs
from .metrics import _bootstrap
from .run import slug

FEWSHOT = [
    ("Paris is the capital of France.", "True"),
    ("Paris is not the capital of France.", "False"),
    ("The Nile is located in Asia.", "False"),
    ("The Nile is not located in Asia.", "True"),
]
INSTRUCTION = ("Decide whether the statement is true or false. "
               "Answer with one word: True or False.")

TRUE_FORMS = ["True", " True", "true", " true", "TRUE"]
FALSE_FORMS = ["False", " False", "false", " false", "FALSE"]


def statement(sentence: str, obj: str) -> str:
    s = sentence.replace(MASK, obj)
    s = s.replace(" .", ".").replace(" ,", ",")  # LAMA templates space out punctuation
    return s[0].upper() + s[1:] if s else s


def build_items(pairs, per_relation: int | None, seed: int = 0):
    """One item per (fact, polarity, object) cell -> 4 items per fact."""
    rng = random.Random(seed)
    by_rel = defaultdict(list)
    for p in pairs:
        by_rel[(p.subset, p.relation)].append(p)

    items = []
    for (subset, relation), ps in by_rel.items():
        objects = sorted({p.obj for p in ps})
        if len(objects) < 2:
            continue  # no distractor available
        if per_relation and len(ps) > per_relation:
            ps = rng.sample(ps, per_relation)
        for p in ps:
            distractor = p.obj
            while distractor == p.obj:
                distractor = rng.choice(objects)
            for polarity, sent in (("affirmative", p.pos), ("negated", p.neg)):
                for kind, obj in (("gold", p.obj), ("distractor", distractor)):
                    # gold is true iff affirmative; distractor is true iff negated
                    truth = (polarity == "affirmative") == (kind == "gold")
                    items.append({
                        "subset": subset, "relation": relation, "uid": p.uid,
                        "sub": p.sub, "obj": obj, "gold_obj": p.obj,
                        "polarity": polarity, "object_kind": kind,
                        "statement": statement(sent, obj), "answer": truth,
                    })
    return items


class TruthScorer:
    def __init__(self, model_name, device="auto", dtype="auto", chat="auto",
                 load_in_4bit=False, trust_remote_code=False):
        self.tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote_code)
        kwargs = dict(trust_remote_code=trust_remote_code)
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
            kwargs["device_map"] = "auto"
        else:
            if dtype == "auto" and not torch.cuda.is_available():
                dtype = "fp32"  # fp16 checkpoints overflow to NaN on CPU/MPS
            kwargs["dtype"] = {"auto": "auto", "fp32": torch.float32,
                               "fp16": torch.float16, "bf16": torch.bfloat16}[dtype]
            if device == "auto" and torch.cuda.is_available():
                kwargs["device_map"] = "auto"
        try:
            self.model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).eval()
        except TypeError:  # transformers < 4.56
            if "dtype" in kwargs:
                kwargs["torch_dtype"] = kwargs.pop("dtype")
            self.model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).eval()
        if not load_in_4bit and device != "auto":
            self.model.to(device)
        self.device = next(self.model.parameters()).device
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.chat = bool(getattr(self.tok, "chat_template", None)) if chat == "auto" else chat == "yes"
        self.true_ids = self._first_ids(TRUE_FORMS)
        self.false_ids = self._first_ids(FALSE_FORMS)

    def _first_ids(self, forms):
        ids = {self.tok.encode(f, add_special_tokens=False)[0] for f in forms
               if self.tok.encode(f, add_special_tokens=False)}
        return sorted(ids)

    def prompt(self, statement: str) -> str:
        if self.chat:
            msg = [{"role": "user", "content": f"{INSTRUCTION}\n\nStatement: {statement}"}]
            return self.tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        shots = "".join(f"Statement: {s}\nAnswer: {a}\n\n" for s, a in FEWSHOT)
        return f"{INSTRUCTION}\n\n{shots}Statement: {statement}\nAnswer:"

    @torch.no_grad()
    def score(self, statements: list[str]):
        """Returns (p_true, p_false) normalised over the True/False tokens."""
        texts = [self.prompt(s) for s in statements]
        enc = [self.tok.encode(t, add_special_tokens=not self.chat) for t in texts]
        L = max(map(len, enc))
        pad = self.tok.pad_token_id
        ids = torch.full((len(enc), L), pad, dtype=torch.long)
        attn = torch.zeros((len(enc), L), dtype=torch.long)
        for i, e in enumerate(enc):  # left padding
            ids[i, L - len(e):] = torch.tensor(e)
            attn[i, L - len(e):] = 1
        ids, attn = ids.to(self.device), attn.to(self.device)
        position_ids = (attn.cumsum(-1) - 1).clamp_min(0)
        try:
            logits = self.model(input_ids=ids, attention_mask=attn,
                                position_ids=position_ids).logits[:, -1].float()
        except TypeError:
            logits = self.model(input_ids=ids, attention_mask=attn).logits[:, -1].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError(f"NaN/inf logits — re-run with --dtype fp32.")
        probs = torch.softmax(logits, dim=-1)
        pt = probs[:, self.true_ids].sum(-1)
        pf = probs[:, self.false_ids].sum(-1)
        total = (pt + pf).clamp_min(1e-12)
        return (pt / total).cpu(), (pf / total).cpu()


def summarize_truth(df: pd.DataFrame) -> pd.DataFrame:
    """Accuracy per cell, negation gap, pair consistency, knowledge-filtered accuracy."""
    rows = []
    for subset, g in df.groupby("subset"):
        acc = g.groupby(["polarity", "object_kind"]).correct.mean()
        aff = g[g.polarity == "affirmative"].correct.mean()
        neg = g[g.polarity == "negated"].correct.mean()

        # consistency: the model flips its verdict between "X ..." and "X not ..."
        piv = g.pivot_table(index=["relation", "uid", "obj"], columns="polarity",
                            values="said_true", aggfunc="first")
        consistency = ((piv.get("affirmative") != piv.get("negated")).mean()
                       if {"affirmative", "negated"} <= set(piv.columns) else float("nan"))

        # only facts the model knows: affirmative+gold answered correctly
        known = g[(g.polarity == "affirmative") & (g.object_kind == "gold") & g.correct]
        keys = set(zip(known.relation, known.uid))
        is_known = pd.Series([k in keys for k in zip(g.relation, g.uid)], index=g.index)
        kn = g[is_known & (g.polarity == "negated")]

        rows.append({
            "subset": subset, "n_items": len(g), "n_facts": g.uid.nunique(),
            "acc_affirmative": aff, "acc_negated": neg, "negation_gap": aff - neg,
            "acc_affirmative_gold": acc.get(("affirmative", "gold"), float("nan")),
            "acc_affirmative_distractor": acc.get(("affirmative", "distractor"), float("nan")),
            "acc_negated_gold": acc.get(("negated", "gold"), float("nan")),
            "acc_negated_distractor": acc.get(("negated", "distractor"), float("nan")),
            "pair_consistency": consistency,
            "acc_negated_known_facts": kn.correct.mean() if len(kn) else float("nan"),
            "says_true_rate": g.said_true.mean(),  # rate of "True" answers (response bias)
            "acc_ci95": _bootstrap(g, "correct"),
        })
    return pd.DataFrame(rows)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--subsets", nargs="*", default=None)
    ap.add_argument("--trex-templates", default=None)
    ap.add_argument("--per-relation", type=int, default=100,
                    help="facts per relation (each gives 4 items); 0 = all")
    ap.add_argument("--chat", default="auto", choices=["auto", "yes", "no"],
                    help="auto: chat template when the tokenizer has one")
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--dtype", default="auto", choices=["auto", "fp32", "fp16", "bf16"])
    ap.add_argument("--load-in-4bit", action="store_true")
    ap.add_argument("--trust-remote-code", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="results_truth")
    args = ap.parse_args(argv)

    pairs = load_pairs(args.data, args.subsets, args.trex_templates)
    items = build_items(pairs, args.per_relation or None, args.seed)
    scorer = TruthScorer(args.model, device=args.device, dtype=args.dtype, chat=args.chat,
                         load_in_4bit=args.load_in_4bit, trust_remote_code=args.trust_remote_code)
    print(f"{args.model}: {len(items)} items "
          f"({'chat template' if scorer.chat else 'few-shot prompt'})")
    print("example prompt:\n" + scorer.prompt(items[0]["statement"]))

    t0 = time.time()
    for i in tqdm(range(0, len(items), args.batch_size)):
        batch = items[i: i + args.batch_size]
        pt, pf = scorer.score([b["statement"] for b in batch])
        for j, b in enumerate(batch):
            b["p_true"] = round(pt[j].item(), 5)
            b["said_true"] = bool(pt[j] > pf[j])
            b["correct"] = int(b["said_true"] == b["answer"])

    df = pd.DataFrame(items)
    out_dir = Path(args.out) / slug(args.model)
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_json(out_dir / "items.jsonl", orient="records", lines=True, force_ascii=False)
    summary = summarize_truth(df)
    summary.insert(0, "model", args.model)
    summary.to_csv(out_dir / "summary.csv", index=False)
    (out_dir / "meta.json").write_text(json.dumps(
        {"model": args.model, "chat": scorer.chat, "n_items": len(df),
         "seconds": round(time.time() - t0, 1), "args": vars(args)}, indent=2))
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print(summary[["subset", "n_items", "acc_affirmative", "acc_negated", "negation_gap",
                       "pair_consistency", "acc_negated_known_facts", "says_true_rate"]].round(3))
    print(f"saved to {out_dir}")


if __name__ == "__main__":
    main()
'''

files['generate.py'] = r'''
"""Experiment D: free generation under negation.

The model is asked to complete a negated sentence. If it fills in the very fact
being denied ("Einstein was not born in ___" -> "Ulm"), it is treating the negated
sentence like the affirmative one. The affirmative version is run as a control:
it measures how often the model knows the fact at all.

    python -m negprobe.generate --model Qwen/Qwen2.5-7B-Instruct --data data/negated_data \
        --subsets TREx --per-relation 50

Reported per subset:
  gold_rate_affirmative  the model completes the affirmative sentence correctly (knowledge)
  gold_rate_negated      it gives the true object even though the sentence denies it (the error)
  error_rate_known       the same, restricted to facts it gets right when affirmative
"""
from __future__ import annotations

import argparse
import json
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from .data import MASK, load_pairs
from .run import slug
from .truth import build_items

PROMPT = ("Complete the sentence with a single plausible answer. "
          "Reply with the answer only.\n\nSentence: {sentence}\nAnswer:")


def normalise(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9 ]+", " ", s.lower()).strip()


def contains_gold(output: str, gold: str) -> bool:
    o, g = normalise(output), normalise(gold)
    return bool(g) and re.search(rf"\b{re.escape(g)}\b", o) is not None


class Generator:
    def __init__(self, model_name, device="auto", dtype="auto", max_new_tokens=12,
                 load_in_4bit=False, trust_remote_code=False):
        self.tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote_code)
        kwargs = dict(trust_remote_code=trust_remote_code)
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
            kwargs["device_map"] = "auto"
        else:
            if dtype == "auto" and not torch.cuda.is_available():
                dtype = "fp32"  # fp16 checkpoints overflow to NaN on CPU/MPS
            kwargs["dtype"] = {"auto": "auto", "fp32": torch.float32,
                               "fp16": torch.float16, "bf16": torch.bfloat16}[dtype]
            if device == "auto" and torch.cuda.is_available():
                kwargs["device_map"] = "auto"
        try:
            self.model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).eval()
        except TypeError:
            if "dtype" in kwargs:
                kwargs["torch_dtype"] = kwargs.pop("dtype")
            self.model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).eval()
        if not load_in_4bit and device != "auto":
            self.model.to(device)
        self.device = next(self.model.parameters()).device
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.tok.padding_side = "left"
        self.chat = bool(getattr(self.tok, "chat_template", None))
        self.max_new_tokens = max_new_tokens

    def prompt(self, sentence: str) -> str:
        text = PROMPT.format(sentence=sentence)
        if self.chat:
            return self.tok.apply_chat_template([{"role": "user", "content": text}],
                                                tokenize=False, add_generation_prompt=True)
        return text

    @torch.no_grad()
    def complete(self, sentences: list[str]) -> list[str]:
        enc = self.tok([self.prompt(s) for s in sentences], return_tensors="pt",
                       padding=True, add_special_tokens=not self.chat).to(self.device)
        out = self.model.generate(**enc, max_new_tokens=self.max_new_tokens,
                                  do_sample=False, pad_token_id=self.tok.pad_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        return [t.strip().split("\n")[0] for t in self.tok.batch_decode(gen, skip_special_tokens=True)]


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--subsets", nargs="*", default=None)
    ap.add_argument("--trex-templates", default=None)
    ap.add_argument("--per-relation", type=int, default=50)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--max-new-tokens", type=int, default=12)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--dtype", default="auto", choices=["auto", "fp32", "fp16", "bf16"])
    ap.add_argument("--load-in-4bit", action="store_true")
    ap.add_argument("--trust-remote-code", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="results_generate")
    args = ap.parse_args(argv)

    pairs = load_pairs(args.data, args.subsets, args.trex_templates)
    # reuse the sampling of Experiment C, but only the gold-object cells
    items = [i for i in build_items(pairs, args.per_relation or None, args.seed)
             if i["object_kind"] == "gold"]
    for it in items:
        it["sentence"] = it["statement"].replace(it["obj"], "___", 1)
        it.pop("statement")

    gen = Generator(args.model, device=args.device, dtype=args.dtype,
                    max_new_tokens=args.max_new_tokens, load_in_4bit=args.load_in_4bit,
                    trust_remote_code=args.trust_remote_code)
    print(f"{args.model}: {len(items)} prompts")
    print("example prompt:\n" + gen.prompt(items[0]["sentence"]))

    t0 = time.time()
    for i in tqdm(range(0, len(items), args.batch_size)):
        batch = items[i: i + args.batch_size]
        for b, o in zip(batch, gen.complete([x["sentence"] for x in batch])):
            b["output"] = o
            b["gold_in_output"] = int(contains_gold(o, b["gold_obj"]))

    df = pd.DataFrame(items)
    out_dir = Path(args.out) / slug(args.model)
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_json(out_dir / "generations.jsonl", orient="records", lines=True, force_ascii=False)

    rows = []
    for subset, g in df.groupby("subset"):
        aff = g[g.polarity == "affirmative"]
        neg = g[g.polarity == "negated"]
        known = set(zip(aff[aff.gold_in_output == 1].relation, aff[aff.gold_in_output == 1].uid))
        neg_known = neg[[k in known for k in zip(neg.relation, neg.uid)]]
        rows.append({
            "model": args.model, "subset": subset, "n_facts": g.uid.nunique(),
            "gold_rate_affirmative": aff.gold_in_output.mean(),
            "gold_rate_negated": neg.gold_in_output.mean(),
            "error_rate_known": neg_known.gold_in_output.mean() if len(neg_known) else float("nan"),
        })
    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "summary.csv", index=False)
    (out_dir / "meta.json").write_text(json.dumps(
        {"model": args.model, "chat": gen.chat, "n_prompts": len(df),
         "seconds": round(time.time() - t0, 1), "args": vars(args)}, indent=2))
    print(summary.round(3).to_string(index=False))
    print("\nexamples where the model answers the denied fact:")
    bad = df[(df.polarity == "negated") & (df.gold_in_output == 1)].head(5)
    for _, r in bad.iterrows():
        print(f"  {r.sentence}  ->  {r.output}   (gold {r.gold_obj})")
    print(f"saved to {out_dir}")


if __name__ == "__main__":
    main()
'''

files['figures.py'] = r'''
"""Turn results into the paper's figures and tables.

    python -m negprobe.figures results --out figures
    python -m negprobe.figures results --truth results_truth --out figures

Produces (PDF for LaTeX + PNG to look at):
  fig1_model_comparison  rho / overlap / Delta per model and subset
  fig2_scaling           metrics against parameter count (Pythia and other families)
  fig3_per_relation      per-relation overlap vs Delta, the spread across relations
  fig4_truth             true/false accuracy by polarity (only with --truth)
  table1_main.tex        the main results table, ready to \\input
  table2_relations.csv   per-relation numbers for the appendix
"""
from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from .metrics import exact_match_p1

# categorical palette, checked for colour-blind separation
INK, MUTED, GRID = "#1A1A1A", "#5A6474", "#D8DCE2"
COLORS = ["#1F6FB2", "#C8761B", "#1E8F7A", "#B23A34"]
MARKERS = ["o", "s", "^", "D"]

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.facecolor": "white",
})

# parameter counts in millions, for the scaling figure
SIZES = {
    "bert-base-cased": 110, "bert-large-cased": 340,
    "roberta-base": 125, "roberta-large": 355,
    "pythia-70m": 70, "pythia-160m": 160, "pythia-410m": 410, "pythia-1b": 1000,
    "pythia-1.4b": 1400, "pythia-2.8b": 2800, "pythia-6.9b": 6900, "pythia-12b": 12000,
    "Qwen2.5-0.5B": 500, "Qwen2.5-1.5B": 1500, "Qwen2.5-3B": 3000, "Qwen2.5-7B": 7000,
    "Qwen2.5-14B": 14000, "Qwen2.5-32B": 32000, "Qwen2.5-72B": 72000,
    "Llama-3.2-1B": 1200, "Llama-3.2-3B": 3200, "Llama-3.1-8B": 8000, "Llama-3.1-70B": 70000,
}


def short(model: str) -> str:
    return model.split("/")[-1]


def params_m(model: str) -> float:
    s = short(model)
    base = s.replace("-Instruct", "").replace("-instruct", "")
    if base in SIZES:
        return SIZES[base]
    m = re.search(r"(\d+(?:\.\d+)?)\s*([bBmM])", base)  # e.g. "-7B", "410m"
    if m:
        return float(m.group(1)) * (1000 if m.group(2).lower() == "b" else 1)
    return float("nan")


def family(model: str) -> str:
    s = short(model).lower()
    for f in ("bert", "roberta", "pythia", "qwen", "llama", "mistral"):
        if f in s:
            return "bert" if s.startswith("bert") else f
    return "other"


def is_instruct(model: str) -> bool:
    return "instruct" in model.lower() or "-it" in model.lower()


def load_summaries(root: Path) -> pd.DataFrame:
    files = sorted(root.glob("*/summary.csv"))
    if not files:
        raise SystemExit(f"no summary.csv under {root}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df["short"] = df.model.map(short)
    df["params_m"] = df.model.map(params_m)
    df["family"] = df.model.map(family)
    df["instruct"] = df.model.map(is_instruct)
    return df


def save(fig, out: Path, name: str):
    for ext in ("pdf", "png"):
        fig.savefig(out / f"{name}.{ext}")
    plt.close(fig)
    print(f"  {name}.pdf / .png")


def fig_model_comparison(df: pd.DataFrame, out: Path):
    metrics = [("spearman_rho", "Rank correlation\naffirmative vs negated"),
               ("top1_overlap", "Same top-1 prediction"),
               ("delta_gold_logp", "Change in log P(correct answer)")]
    subsets = sorted(df.subset.unique())
    models = list(dict.fromkeys(df.sort_values("params_m")["short"]))
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
    y = np.arange(len(models))
    h = 0.8 / max(len(subsets), 1)
    for ax, (col, title) in zip(axes, metrics):
        for k, sub in enumerate(subsets):
            d = df[df.subset == sub].set_index("short").reindex(models)[col]
            ax.barh(y + k * h - 0.4 + h / 2, d.values, height=h * 0.86,
                    color=COLORS[k], label=sub, zorder=3)
        ax.set_yticks(y, models if ax is axes[0] else [""] * len(models))
        ax.set_title(title, loc="left")
        ax.xaxis.grid(True, zorder=0)
        ax.invert_yaxis()
        if col == "delta_gold_logp":
            ax.axvline(0, color=MUTED, lw=0.8)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper right", bbox_to_anchor=(0.99, 1.06),
               ncol=len(subsets), fontsize=8)
    fig.suptitle("Negation sensitivity by model", x=0.02, y=1.06, ha="left", fontsize=11)
    save(fig, out, "fig1_model_comparison")


def fig_scaling(df: pd.DataFrame, out: Path):
    d = df.dropna(subset=["params_m"])
    if d.empty:
        return
    metrics = [("top1_overlap", "Same top-1 prediction"),
               ("delta_gold_logp", "Change in log P(correct answer)"),
               ("p@1_affirmative", "P@1, affirmative")]
    subsets = sorted(d.subset.unique())
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharex=True)
    fams = sorted(d.family.unique())
    for ax, (col, title) in zip(axes, metrics):
        for i, fam in enumerate(fams):
            for k, sub in enumerate(subsets):
                s = d[(d.family == fam) & (d.subset == sub)].sort_values("params_m")
                if s.empty:
                    continue
                style = "--" if sub != subsets[0] else "-"
                ax.plot(s.params_m, s[col], style, color=COLORS[i % len(COLORS)],
                        marker=MARKERS[k % len(MARKERS)], ms=5, lw=1.6, zorder=3,
                        label=f"{fam} · {sub}")
        ax.set_xscale("log")
        ax.set_xlabel("parameters (millions, log scale)")
        ax.set_title(title, loc="left")
        ax.grid(True, zorder=0)
        if col == "delta_gold_logp":
            ax.axhline(0, color=MUTED, lw=0.8)
    axes[-1].legend(fontsize=7, loc="best")
    fig.suptitle("Does scale change how models handle negation?", x=0.02, y=1.06,
                 ha="left", fontsize=11)
    save(fig, out, "fig2_scaling")


def fig_per_relation(root: Path, out: Path, subset: str = "TREx"):
    frames = []
    for f in sorted(list(root.glob("*/pairs.jsonl")) + list(root.glob("*/pairs.jsonl.gz"))):
        d = exact_match_p1(pd.read_json(f, lines=True))
        d = d[d.subset == subset]
        if d.empty:
            continue
        g = d.groupby("relation")[["overlap", "delta", "p1_pos"]].mean()
        g["model"] = short(d.model.iloc[0])
        frames.append(g.reset_index())
    if not frames:
        return
    rel = pd.concat(frames)
    models = sorted(rel.model.unique())
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for i, m in enumerate(models):
        s = rel[rel.model == m]
        ax.scatter(s.delta, s.overlap, s=26, color=COLORS[i % len(COLORS)],
                   edgecolor="white", linewidth=0.8, label=m, zorder=3)
    worst = rel[rel.model == models[0]].nlargest(3, "overlap")
    for _, r in worst.iterrows():
        ax.annotate(r.relation, (r.delta, r.overlap), fontsize=7, color=MUTED,
                    xytext=(4, 4), textcoords="offset points")
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.set_xlabel("change in log P(correct answer) under negation")
    ax.set_ylabel("share with the same top-1 prediction")
    ax.set_title(f"Negation sensitivity varies by relation ({subset})", loc="left")
    ax.grid(True, zorder=0)
    ax.legend(fontsize=8)
    save(fig, out, "fig3_per_relation")


def fig_truth(truth_root: Path, out: Path):
    """One row per model and subset. Values are exactly those in summary.csv (no averaging
    across subsets), so the figure matches the tables."""
    files = sorted(truth_root.glob("*/summary.csv"))
    if not files:
        return
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df["row"] = df.model.map(short) + " · " + df.subset
    df = df.sort_values(["model", "subset"], ascending=[True, False])
    rows = list(df["row"])
    fig, axes = plt.subplots(1, 2, figsize=(9, 0.9 + 0.9 * len(rows)))
    y = np.arange(len(rows))
    for k, (col, lab) in enumerate([("acc_affirmative", "affirmative"),
                                    ("acc_negated", "negated")]):
        axes[0].barh(y + k * 0.4 - 0.2, df[col].values, height=0.34, color=COLORS[k],
                     label=lab, zorder=3)
    axes[0].axvline(0.5, color=MUTED, lw=0.8, ls=":")
    axes[0].set_yticks(y, rows)
    axes[0].set_xlim(0, 1.0)  # room for the legend to the right of the bars
    axes[0].set_xlabel("accuracy (0.5 = chance)")
    axes[0].set_title("True/false judgements", loc="left")
    axes[0].legend(fontsize=8, loc="lower right")
    axes[0].invert_yaxis()
    axes[0].xaxis.grid(True, zorder=0)

    for k, col in enumerate(["pair_consistency", "says_true_rate"]):
        axes[1].barh(y + k * 0.4 - 0.2, df[col].values, height=0.34,
                     color=COLORS[k + 2], label=col.replace("_", " "), zorder=3)
    axes[1].axvline(0.5, color=MUTED, lw=0.8, ls=":")
    axes[1].set_yticks(y, [""] * len(rows))
    axes[1].set_xlim(0, 0.9)
    axes[1].set_xlabel("share of statement pairs / of items")
    axes[1].set_title('Consistency and rate of "True" answers', loc="left")
    axes[1].legend(fontsize=8, loc="lower right")
    axes[1].invert_yaxis()
    axes[1].xaxis.grid(True, zorder=0)
    save(fig, out, "fig4_truth")


def tables(df: pd.DataFrame, root: Path, out: Path):
    cols = ["short", "subset", "n_pairs", "spearman_rho", "top1_overlap",
            "delta_gold_logp", "p@1_affirmative", "p@1_negated"]
    t = df[cols].sort_values(["subset", "short"]).round(3)
    t.columns = ["Model", "Data", "N", r"$\rho$", "Overlap", r"$\Delta$", "P@1", "P@1 (neg)"]
    (out / "table1_main.tex").write_text(t.to_latex(
        index=False, escape=False, column_format="llrrrrrr",
        caption="Negation probing results. $\\rho$ and overlap compare the affirmative and "
                "negated distributions; $\\Delta$ is the change in the log probability of the "
                "correct answer under negation.",
        label="tab:main"))
    frames = []
    for f in sorted(list(root.glob("*/pairs.jsonl")) + list(root.glob("*/pairs.jsonl.gz"))):
        d = exact_match_p1(pd.read_json(f, lines=True))
        g = (d.groupby(["subset", "relation"])[["rho", "overlap", "delta", "p1_pos", "p1_neg"]]
             .mean().round(3).reset_index())
        g.insert(0, "model", short(d.model.iloc[0]))
        frames.append(g)
    if frames:
        pd.concat(frames).to_csv(out / "table2_relations.csv", index=False)
    print("  table1_main.tex / table2_relations.csv")


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("results", help="folder with one sub-folder per model (Experiments A+B)")
    ap.add_argument("--truth", default=None, help="results folder of Experiment C")
    ap.add_argument("--subset", default="TREx", help="subset for the per-relation figure")
    ap.add_argument("--out", default="figures")
    args = ap.parse_args(argv)

    root, out = Path(args.results), Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    df = load_summaries(root)
    print(f"{df.model.nunique()} models, {len(df)} rows. writing to {out}/")
    fig_model_comparison(df, out)
    fig_scaling(df, out)
    fig_per_relation(root, out, args.subset)
    if args.truth:
        fig_truth(Path(args.truth), out)
    tables(df, root, out)
    (out / "figures_meta.json").write_text(json.dumps(
        {"models": sorted(df.model.unique()), "subsets": sorted(df.subset.unique())}, indent=2))


if __name__ == "__main__":
    main()
'''

files['compare.py'] = r'''
"""Merge summary.csv files of several models into one table.

    python -m negprobe.compare results --out results/all_models.csv
"""
import argparse
from pathlib import Path

import pandas as pd


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("results", help="folder with one sub-folder per model")
    ap.add_argument("--out", default=None)
    args = ap.parse_args(argv)
    files = sorted(Path(args.results).glob("*/summary.csv"))
    if not files:
        raise SystemExit(f"no summary.csv found under {args.results}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    cols = ["model", "subset", "n_pairs", "spearman_rho", "top1_overlap",
            "delta_gold_logp", "gold_rank_dropped", "p@1_affirmative", "p@1_negated",
            "spearman_rho_micro", "spearman_rho_micro_ci95",
            "delta_gold_logp_micro", "delta_micro_ci95"]
    table = df[cols].sort_values(["subset", "model"])
    out = args.out or Path(args.results) / "all_models.csv"
    table.to_csv(out, index=False)
    with pd.option_context("display.width", 220, "display.max_columns", 30):
        print(table.round(3).to_string(index=False))
    print(f"saved to {out}")


if __name__ == "__main__":
    main()
'''

for n, t in files.items():
    pathlib.Path('negprobe/negprobe/' + n).write_text(t.lstrip('\n'))
print('wrote', len(files), 'modules')

## Data

In [ ]:
%cd /content/negprobe
!mkdir -p data && cd data && \
  (test -f negated_data.tar.gz || wget -q https://dl.fbaipublicfiles.com/LAMA/negated_data.tar.gz) && \
  mkdir -p negated_data && tar -xzf negated_data.tar.gz -C negated_data && \
  (test -f common_vocab_cased.txt || wget -q https://dl.fbaipublicfiles.com/LAMA/common_vocab_cased.txt)
!ls data/negated_data

## Runs reported in the paper
Run one cell at a time and **download after each** — free runtimes disconnect. Runtimes on a T4: 8.3, 13.1, 15.4 and 4.8 min.

In [ ]:
!python -m negprobe.truth --model Qwen/Qwen2.5-1.5B-Instruct --data data/negated_data --subsets TREx GoogleRE --per-relation 50 --dtype fp32 --batch-size 16 --out results_truth

In [ ]:
!python -m negprobe.truth --model Qwen/Qwen2.5-1.5B --data data/negated_data --subsets TREx GoogleRE --per-relation 50 --chat no --dtype fp32 --batch-size 16 --out results_truth

In [ ]:
!python -m negprobe.run --model EleutherAI/pythia-1.4b --data data/negated_data --vocab data/common_vocab_cased.txt --subsets GoogleRE TREx --dtype fp32 --batch-size 32 --out results

In [ ]:
!python -m negprobe.run --model EleutherAI/pythia-410m --data data/negated_data --vocab data/common_vocab_cased.txt --subsets GoogleRE TREx --dtype fp32 --batch-size 64 --out results

In [ ]:
# optional, not in the paper: Experiment D (free generation)
!python -m negprobe.generate --model Qwen/Qwen2.5-1.5B-Instruct --data data/negated_data --subsets TREx --per-relation 50 --dtype fp32 --batch-size 8

## Download results (run after every model)

In [ ]:
!zip -qr /content/negres.zip results results_truth results_generate 2>/dev/null; du -h /content/negres.zip
from google.colab import files; files.download('/content/negres.zip')